<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/SPARQL%20for%20Graph%20Querying/09b-lab-sparql-by-hand-three-graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9b — Reading RDF Connections, Then Writing SPARQL

Most people who get stuck writing SPARQL are not stuck on the syntax. They are stuck because they **cannot see how the triples connect to each other** — so they don't know what path to ask for. This lab fixes that first.

The whole idea in one sentence:

> **RDF is a graph of connections. To answer a question you trace a path of connections by hand. A SPARQL query is just that path, written down.**

So every exercise runs the same loop, and step 2 is the one that matters most:

1. **Read the question.**
2. **Trace the connections by hand.** Find the starting node, follow the arrows (each `subject predicate object` triple is one arrow), and note which node you land on. *Write the path down as nodes and arrows before anything else.*
3. **Read off your answer** from where the path lands.
4. **Turn the path into SPARQL** — one arrow you followed becomes one triple pattern — run it, and check it matches your hand answer.

There is a collapsed **Solution** under every question showing the traced path, the hand answer, and the query. Don't open it until you've drawn your own path.

**Three graphs, getting harder — and each one small enough to fit on a single slide and work out by hand:**

| Part | Graph | Size | The connection skill you build |
|---|---|---|---|
| A | A tiny **bookshop** | 3 books | one arrow; then *two arrows through a shared node* (the join) |
| B | **University enrolments** | 4 students, 3 courses | a middle "link node" that joins two things and holds extra facts |
| C | A **family tree** | 7 people | following the *same* arrow many times — chains of connections |

---

## First: how to read RDF as connections

An RDF file is a list of **triples**. Every triple is one arrow in a graph:

```
subject  ─predicate→  object
```

Read `ex:b1 ex:title "Clean Code"` as: *the node `ex:b1` has an arrow labelled `ex:title` pointing to `"Clean Code"`.*

Two kinds of object sit at the end of those arrows, and telling them apart is the key skill:

- A **literal** (in quotes, or a number) — `"Clean Code"`, `42`. This is a dead end: a value, nothing points onward from it.
- A **URI** — `ex:b1`, `ex:martin`. This is a **node you can keep travelling from**, because it may be the *subject* of other triples.

**Nodes connect when the same URI appears in two triples.** Look at these two lines:

```
ex:b1     ex:author  ex:martin .       ← ex:martin is the OBJECT here
ex:martin ex:name    "Robert Martin" . ← ex:martin is the SUBJECT here
```

`ex:martin` is the landing point of the first arrow and the launch point of the second. That shared node is a **connection**: it lets you travel `ex:b1 → ex:martin → "Robert Martin"` and discover that book `ex:b1`'s author is named Robert Martin — even though no single triple says so.

**This is exactly what a SPARQL join is.** When you write

```sparql
?b ex:author ?author .
?author ex:name ?name .
```

the repeated `?author` is you telling the engine: *the node I land on in line 1 is the same node I leave from in line 2* — the same hand-off you did by finger. Every join you will ever write is just "the same node, reached from two sides."

Keep this table next to you:

| In the RDF you see… | It means… | When you trace / query… |
|---|---|---|
| `A pred B` | an arrow `A ─pred→ B` | follow it forward: from `A` you reach `B` |
| the same URI as object here and subject there | a connection between two arrows | hand off from one arrow to the next through that node |
| a literal object (quotes / number) | a value, a dead end | you've arrived — read it off |
| a variable `?x` reused in two patterns | "must be the same node" | that's the join; it welds the two arrows together |

Now let's read some graphs.

## Setup

Run once. Installs the `%%sparql` cell magic (from the course's `cellspell` package) and `rdflib`. Each query cell starts with `%%sparql --file <name>.ttl` and runs against that Turtle file.

In [ ]:
# ── Install the SPARQL cell magic + rdflib ──────────────
!pip install "cellspell[sparql] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
!pip install -q rdflib
%load_ext cellspell.sparql

---

## Part A — The Bookshop  🟢 *one arrow, then two*

Three books — the whole graph fits on a slide. Read it as arrows. A book node has arrows for `ex:title`, `ex:price`, `ex:topic` (all landing on **literals** — dead ends), and one `ex:author` arrow landing on a **person URI** — a node you can travel onward from to reach the author's name.

Run the next cell to write `bookshop.ttl`, then read it as a graph, not as text.

In [ ]:
%%writefile bookshop.ttl
@prefix ex:  <http://example.org/book#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:b1 rdf:type ex:Book ; ex:title "Clean Code" ;
      ex:author ex:martin ; ex:price 42 ; ex:topic "Software" .

ex:b2 rdf:type ex:Book ; ex:title "SQL for Smarties" ;
      ex:author ex:celko ; ex:price 55 ; ex:topic "Databases" .

ex:b3 rdf:type ex:Book ; ex:title "Designing Data-Intensive Applications" ;
      ex:author ex:kleppmann ; ex:price 45 ; ex:topic "Databases" .

ex:martin     ex:name "Robert Martin" .
ex:celko      ex:name "Joe Celko" .
ex:kleppmann  ex:name "Martin Kleppmann" .

**Warm-up — draw the connections (no SPARQL yet).** For book `ex:b1`, list every arrow leaving it and say where each lands. Which arrows end at a dead-end literal, and which one lands on a node you can keep travelling from?

<details>
<summary>Check yourself</summary>

`ex:b1` has five arrows out:

```
ex:b1 ─rdf:type→ ex:Book
ex:b1 ─ex:title→ "Clean Code"      (literal — dead end)
ex:b1 ─ex:author→ ex:martin        (URI — travel onward!)
ex:b1 ─ex:price→ 42                (literal — dead end)
ex:b1 ─ex:topic→ "Software"        (literal — dead end)
```

Only `ex:author` lands on a node (`ex:martin`) with its own outgoing arrow: `ex:martin ─ex:name→ "Robert Martin"`. So to get the author's *name* from a book you must make **two** hops, handing off through `ex:martin`. Every other fact is one hop away.

</details>

**Standard prefix for Part A** — paste at the top of each query cell:

```sparql
PREFIX ex:  <http://example.org/book#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
```

### A1 — List every book title  *(follow one arrow)*

**Trace the connections.** From each book node, follow the single `ex:title` arrow. Where does it land? How many books are there, so how many arrows, so how many rows?

**Now write it.** One arrow you followed = one triple pattern. Bind the book with `?b` and the landing literal with `?title`.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

# One arrow: (some book) --ex:title--> ?title
SELECT ?title WHERE {
    # your triple pattern here
}

<details>
<summary>Solution</summary>

**Path traced:** `?b ─ex:title→ ?title`, for every book node. Three book nodes → three arrows → **3 rows**: *Clean Code, SQL for Smarties, Designing Data-Intensive Applications.*

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title WHERE {
    ?b ex:title ?title .
}
```

The single pattern *is* the single arrow. `?b` isn't in the SELECT but it still does the work: it insists `?title` be reached *from a book node*, not from anywhere.

</details>

### A2 — Databases books under £50  *(one node, several arrows, then a test)*

**Trace the connections.** Start only from book nodes whose `ex:topic` arrow lands on `"Databases"`. From each survivor, also follow its `ex:title` and `ex:price` arrows. Finally test the price you landed on: keep it only if `< 50`.

**Now write it.** Three arrows leaving the *same* book node (so, same subject `?b`), then a `FILTER` on the price literal.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

SELECT ?title ?price WHERE {
    # same book node: --ex:topic--> "Databases", --ex:title--> ?title, --ex:price--> ?price
    # then FILTER the price
}

<details>
<summary>Solution</summary>

**Path traced:** the two nodes with a `Databases` topic arrow are `ex:b2` (SQL for Smarties, 55) and `ex:b3` (Designing Data-Intensive Applications, 45). Test `price < 50`: b2 fails, b3 passes → **1 row**.

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title ?price WHERE {
    ?b ex:topic "Databases" ;
       ex:title ?title ;
       ex:price ?price .
    FILTER(?price < 50)
}
```

| ?title | ?price |
|---|---|
| Designing Data-Intensive Applications | 45 |

The `;` keeps the same subject `?b` — that's three arrows leaving **one** node, which is why title, price and topic all describe the same book. `FILTER` runs after the arrows are matched, testing the literal you landed on.

</details>

### A3 — Each title with its author's name  *(the two-arrow hand-off = the join)*

This is the one from the primer. The author's **name** is not on the book. You go: book `─ex:author→` person node `─ex:name→` name. The person node is the hand-off.

**Trace the connections.** Take `ex:b1`: follow `ex:author` → you land on `ex:martin`. Now *leave* `ex:martin` along `ex:name` → `"Robert Martin"`. Do that for all three books and write the *(title, name)* pairs.

**Now write it.** Two patterns. The landing node of arrow 1 is the launch node of arrow 2 — so **use the same variable** `?author` in both. That shared variable *is* the connection.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

SELECT ?title ?name WHERE {
    # arrow 1: ?b --ex:title--> ?title  and  ?b --ex:author--> ?author
    # arrow 2: ?author --ex:name--> ?name       (SAME ?author = the hand-off)
}

<details>
<summary>Solution</summary>

**Path traced** (for each book): `?b ─ex:author→ ?author ─ex:name→ ?name`, and separately `?b ─ex:title→ ?title`. Three books, each with one author, each author with one name → **3 rows**:

| ?title | ?name |
|---|---|
| Clean Code | Robert Martin |
| SQL for Smarties | Joe Celko |
| Designing Data-Intensive Applications | Martin Kleppmann |

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title ?name WHERE {
    ?b ex:title ?title ;
       ex:author ?author .
    ?author ex:name ?name .
}
```

`?author` appears as the **object** of `ex:author` and the **subject** of `ex:name` — exactly the "same URI in two triples" pattern from the primer. Repeating the variable tells the engine *these two arrows meet at the same node*. That is the entire mechanism of a join; everything harder is more of this.

</details>

---

## Part B — University Enrolments  🟡 *the middle "link node"*

In Part A the hand-off node (`ex:martin`) was a person. Here the hand-off node is a **relationship turned into a node**: an enrolment (`ex:e1` … `ex:e6`) sits *between* a student and a course, with arrows `ex:student` and `ex:course` leaving it. Why bother? Because a plain arrow student→course couldn't also carry a **grade** — but a node can. So each enrolment node connects three things: a student, a course, and a grade.

Reading direction matters: to get from a student to their courses you travel **into** an enrolment node backwards along `ex:student`, then **out** along `ex:course`.

Two deliberate gaps to spot while reading — the whole graph still fits on a slide, so hunt for them: **Carol's AI enrolment (`ex:e5`) has no `ex:grade` arrow**, and **Dan has no enrolment node pointing at him at all**. In RDF a missing arrow means "not stated" (open-world) — not "zero", not "false".

In [ ]:
%%writefile enrolments.ttl
@prefix ex:  <http://example.org/uni#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

# ── Students ──
ex:alice rdf:type ex:Student ; ex:name "Alice" .
ex:bob   rdf:type ex:Student ; ex:name "Bob" .
ex:carol rdf:type ex:Student ; ex:name "Carol" .
ex:dan   rdf:type ex:Student ; ex:name "Dan" .

# ── Courses ──
ex:db  rdf:type ex:Course ; ex:cname "Databases" .
ex:web rdf:type ex:Course ; ex:cname "Web Systems" .
ex:ai  rdf:type ex:Course ; ex:cname "AI" .

# ── Enrolments: each is a LINK NODE joining a student to a course (grade optional) ──
ex:e1 ex:student ex:alice ; ex:course ex:db  ; ex:grade 72 .
ex:e2 ex:student ex:bob   ; ex:course ex:db  ; ex:grade 58 .
ex:e3 ex:student ex:carol ; ex:course ex:db  ; ex:grade 77 .
ex:e4 ex:student ex:carol ; ex:course ex:web ; ex:grade 90 .
ex:e5 ex:student ex:carol ; ex:course ex:ai  .            # no grade yet
ex:e6 ex:student ex:bob   ; ex:course ex:ai  ; ex:grade 65 .
# Dan: a Student with NO enrolment node pointing at him.

**Warm-up — trace one link node.** Take `ex:e1`. What arrows leave it, and what does each connect to? Draw the little star of arrows around `ex:e1`.

<details>
<summary>Check yourself</summary>

```
        ex:alice ←─ex:student─ ex:e1 ─ex:course→ ex:db
                                 │
                              ex:grade
                                 ↓
                                72
```

`ex:e1` is the hub. Reading a student's grade in Databases means arriving at this hub from Alice (backwards along `ex:student`), confirming its `ex:course` points to `ex:db`, and reading its `ex:grade`. The hub is what lets one grade belong to *this specific* student-course pair.

</details>

**Standard prefix for Part B:**

```sparql
PREFIX ex:  <http://example.org/uni#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
```

### B1 — Who is enrolled in Databases?  *(course ← link node → student → name)*

**Trace the connections.** Find every enrolment node whose `ex:course` arrow lands on `ex:db`. From each such node, follow `ex:student` to a student node, then that student's `ex:name` to a literal. Three hand-offs.

**Now write it.** The enrolment node `?e` is the pivot; `?s` is the student it hands off to.

In [ ]:
%%sparql --file enrolments.ttl
PREFIX ex:  <http://example.org/uni#>

SELECT ?sname WHERE {
    # ?e --ex:course--> ex:db ;  ?e --ex:student--> ?s ;  ?s --ex:name--> ?sname
}

<details>
<summary>Solution</summary>

**Path traced:** enrolments pointing at `ex:db` are `e1` (→Alice), `e2` (→Bob), `e3` (→Carol) → **3 rows**: Alice, Bob, Carol.

**Query:**

```sparql
PREFIX ex: <http://example.org/uni#>
SELECT ?sname WHERE {
    ?e ex:course ex:db ;
       ex:student ?s .
    ?s ex:name ?sname .
}
```

Same shared-node idea as A3, one hop longer: `?e` connects the course test to the student arrow, then `?s` connects the student arrow to the name arrow. Two hand-offs, two shared variables.

</details>

### B2 — All of Carol's enrolments, grade if it exists  *(a missing arrow)*

Carol has three enrolment nodes, but one (AI, `ex:e5`) has **no `ex:grade` arrow leaving it**. If you *require* a grade arrow, that enrolment falls out of the path and you lose the row. `OPTIONAL` means "follow this arrow if it exists, otherwise still keep the row, just leave the value blank."

**Trace the connections.** From Carol, reach each enrolment (backwards along `ex:student`), follow `ex:course` → `ex:cname` for the course name, and *try* to follow `ex:grade`. For AI there is no grade arrow — mark it "—".

**Now write it.** Wrap the grade arrow in `OPTIONAL { }`.

In [ ]:
%%sparql --file enrolments.ttl
PREFIX ex:  <http://example.org/uni#>

SELECT ?cname ?grade WHERE {
    # ?e --ex:student--> ex:carol ; ?e --ex:course--> ?c ; ?c --ex:cname--> ?cname
    # OPTIONAL: ?e --ex:grade--> ?grade
}

<details>
<summary>Solution</summary>

**Path traced:** Carol's enrolments are `e3`(DB,77), `e4`(Web,90), `e5`(AI, *no grade arrow*) → **3 rows**:

| ?cname | ?grade |
|---|---|
| Databases | 77 |
| Web Systems | 90 |
| AI | *(blank)* |

**Query:**

```sparql
PREFIX ex: <http://example.org/uni#>
SELECT ?cname ?grade WHERE {
    ?e ex:student ex:carol ;
       ex:course ?c .
    ?c ex:cname ?cname .
    OPTIONAL { ?e ex:grade ?grade . }
}
```

Drop the `OPTIONAL` and the AI row vanishes — because the *required* grade arrow doesn't exist for `e5`, the whole path through `e5` breaks. `OPTIONAL` is a **left join**: keep the arrows you did follow, blank out the one that wasn't there. The empty cell is the open-world assumption you can see.

</details>

### B3 — Enrolment count per course, most popular first  *(gather connections, then count)*

**Trace the connections.** For each course, count how many enrolment nodes point at it via `ex:course`. Databases ← which enrolments? Web Systems? AI? Rank them.

**Now write it.** Follow enrolment `─ex:course→` course `─ex:cname→` name, then `GROUP BY` the course name and `COUNT` the enrolments feeding each group.

In [ ]:
%%sparql --file enrolments.ttl
PREFIX ex:  <http://example.org/uni#>

SELECT ?cname (COUNT(?e) AS ?n) WHERE {
    # ?e --ex:course--> ?c --ex:cname--> ?cname
}
# GROUP BY ... ORDER BY ...

<details>
<summary>Solution</summary>

**Path traced / tallied:** Databases ← e1, e2, e3 = 3. AI ← e5, e6 = 2. Web Systems ← e4 = 1.

**Query:**

```sparql
PREFIX ex: <http://example.org/uni#>
SELECT ?cname (COUNT(?e) AS ?n) WHERE {
    ?e ex:course ?c .
    ?c ex:cname ?cname .
}
GROUP BY ?cname
ORDER BY DESC(?n) ?cname
```

| ?cname | ?n |
|---|---|
| Databases | 3 |
| AI | 2 |
| Web Systems | 1 |

You first *found all the connections* (every enrolment→course arrow), then `GROUP BY` bucketed them by course and `COUNT` measured each bucket. The secondary `?cname` in `ORDER BY` keeps the output stable when counts tie.

</details>

### B4 — Students with no enrolment  *(a node with no arrow coming in)*

**Trace the connections.** Go through the four students. For each, ask: does *any* enrolment node have an `ex:student` arrow landing on them? Alice, Bob, Carol — yes. Dan — no arrow arrives. Dan is the answer.

**Now write it.** List every student, then keep only those where the incoming-enrolment connection is **absent**, using `FILTER NOT EXISTS`.

In [ ]:
%%sparql --file enrolments.ttl
PREFIX ex:  <http://example.org/uni#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?sname WHERE {
    ?s rdf:type ex:Student ; ex:name ?sname .
    # FILTER NOT EXISTS { some enrolment --ex:student--> ?s }
}

<details>
<summary>Solution</summary>

**Path traced:** an `ex:student` arrow lands on Alice, Bob, Carol. None lands on **Dan** → **1 row**: Dan.

**Query:**

```sparql
PREFIX ex: <http://example.org/uni#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?sname WHERE {
    ?s rdf:type ex:Student ;
       ex:name ?sname .
    FILTER NOT EXISTS { ?e ex:student ?s . }
}
```

Everything so far has asked "which connections exist?" This asks the mirror question: "which node has **no** connection of this kind coming in?" `FILTER NOT EXISTS` is the only way RDF lets you talk about a missing arrow — you can test for absence, but you can't invent the fact that isn't there.

</details>

---

## Part C — The Family Tree  🔴 *chains of the same connection*

One arrow type, `ex:parentOf` (parent → child), repeated to build a tree — seven people, still a single slide. Everything here is about **following the same connection several times in a row**. Grandchild = parentOf twice. Descendant = parentOf any number of times. That repetition is what *property paths* express, so you don't have to guess the depth up front.

The tree (parents above children):

```
        Ada ─┬─ Ben
             │
       ┌─────┴─────┐
      Cara        Dev
     ┌──┴──┐       │
    Eli   Fay     Gus
```

Read every `ex:parentOf` line of the file as one `parent ─parentOf→ child` arrow and check it against this picture.

In [ ]:
%%writefile family.ttl
@prefix ex:   <http://example.org/fam#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:ada  foaf:name "Ada" .   ex:ben  foaf:name "Ben" .
ex:cara foaf:name "Cara" .  ex:dev  foaf:name "Dev" .
ex:eli  foaf:name "Eli" .   ex:fay  foaf:name "Fay" .
ex:gus  foaf:name "Gus" .

# ex:parentOf : parent -> child  (one arrow type, used seven times)
ex:ada  ex:parentOf ex:cara .   ex:ada ex:parentOf ex:dev .
ex:ben  ex:parentOf ex:cara .   ex:ben ex:parentOf ex:dev .
ex:cara ex:parentOf ex:eli .    ex:cara ex:parentOf ex:fay .
ex:dev  ex:parentOf ex:gus .

**Warm-up — read arrows, rebuild the tree.** Cover the diagram. Using only the seven `ex:parentOf` lines, write who each person's children are. Then uncover and check. (This *is* the skill — the diagram is just those seven arrows drawn.)

**Standard prefix for Part C:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
```

### C1 — Ada's grandchildren  *(follow the same arrow twice)*

Grandchild = follow `ex:parentOf` from Ada to a child, then `ex:parentOf` again from that child. The path `ex:parentOf/ex:parentOf` means "these two arrows in a row"; the middle child node is passed through automatically.

**Trace the connections.** Ada `─parentOf→` Cara, Dev. Cara `─parentOf→` Eli, Fay. Dev `─parentOf→` Gus. Two hops from Ada lands you on…?

**Now write it.** Use the sequence path `/`. Notice you don't name the middle child — the path hides it, just like you didn't write it down when tracing.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?gname WHERE {
    # ex:ada --ex:parentOf--> (child) --ex:parentOf--> ?gc ;  ?gc --foaf:name--> ?gname
}

<details>
<summary>Solution</summary>

**Path traced:** Ada→{Cara,Dev}→{Eli,Fay,Gus}. **3 rows**: Eli, Fay, Gus.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?gname WHERE {
    ex:ada ex:parentOf/ex:parentOf ?gc .
    ?gc foaf:name ?gname .
}
```

`ex:parentOf/ex:parentOf` is "two arrows in a row." The intermediate child is matched inside the path and never becomes a variable — exactly how, when tracing by hand, you glide through the child to reach the grandchild. (Writing it the long way with an explicit `?child` in two patterns gives the same answer.)

</details>

### C2 — All of Ada's descendants  *(follow the same arrow any number of times)*

Now the depth is open: children, grandchildren, and further if the tree were taller. `ex:parentOf+` means "follow `ex:parentOf` **one or more** times" — keep travelling down until the arrows run out.

**Trace the connections.** From Ada, sweep down the whole subtree collecting everyone you can reach by parentOf arrows.

**Now write it.** Use `+`. Compare with C1: `/` was *exactly two* hops; `+` is *one or more* — the query you literally cannot write with a fixed number of patterns because you don't know the depth.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?dname WHERE {
    # ex:ada --ex:parentOf+--> ?d ;  ?d --foaf:name--> ?dname
}

<details>
<summary>Solution</summary>

**Path traced:** children Cara, Dev; then their children Eli, Fay, Gus. **5 rows**: Cara, Dev, Eli, Fay, Gus.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?dname WHERE {
    ex:ada ex:parentOf+ ?d .
    ?d foaf:name ?dname .
}
```

`+` walks the connection repeatedly and returns every node reachable downward — the whole subtree. (`*` would be *zero* or more, which also returns Ada herself.) This is the pay-off of thinking in connections: "descendant" isn't a new predicate in the data, it's just "the parentOf arrow, followed as far as it goes."

</details>

### C3 — Cara's siblings  *(two arrows out of the same parent)*

Siblings share a parent. So the connection is: land on a parent of Cara, then leave that *same* parent along `ex:parentOf` again to reach the other children.

**Trace the connections.** Who points `parentOf` at Cara? Ada and Ben. From Ada, the parentOf arrows go to Cara and Dev; from Ben, to Cara and Dev. Remove Cara herself → the sibling is…?

**Now write it.** The shared node is the **parent** `?p`, used as the subject of two `ex:parentOf` arrows (a self-join). Add `FILTER(?sib != ex:cara)` to drop Cara, and `DISTINCT` because Dev is reached through *both* parents.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT DISTINCT ?sibname WHERE {
    # ?p --ex:parentOf--> ex:cara   AND   ?p --ex:parentOf--> ?sib   (same ?p)
    # exclude Cara ; then ?sib --foaf:name--> ?sibname
}

<details>
<summary>Solution</summary>

**Path traced:** parents of Cara = {Ada, Ben}; each parent's children = {Cara, Dev}; drop Cara → **1 row**: Dev. Dev shows up via Ada *and* via Ben, so without `DISTINCT` he'd appear twice.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT DISTINCT ?sibname WHERE {
    ?p ex:parentOf ex:cara .
    ?p ex:parentOf ?sib .
    FILTER(?sib != ex:cara)
    ?sib foaf:name ?sibname .
}
```

`?p` is the shared node used **twice as a launch point** — a self-join on `ex:parentOf`. Delete `DISTINCT`, re-run, and you'll see Dev twice: two *routes* (through Ada, through Ben) to the *same* sibling. Counting routes vs. counting people is the trap `DISTINCT` guards.

</details>

### C4 — The roots of the tree  *(no arrow coming in)*

**Trace the connections.** For each person, ask: does any `ex:parentOf` arrow land on them? Everyone has an incoming parent arrow except the two at the very top.

**Now write it.** Same absence test as B4, but this time the missing arrow is *incoming* — the person sits in the **object** slot.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE {
    ?person foaf:name ?name .
    # FILTER NOT EXISTS { someone --ex:parentOf--> ?person }
}

<details>
<summary>Solution</summary>

**Path traced:** an incoming `parentOf` arrow reaches everyone except **Ada** and **Ben** → **2 rows**.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ?person foaf:name ?name .
    FILTER NOT EXISTS { ?anyone ex:parentOf ?person . }
}
```

A "root" is just a node with no `ex:parentOf` connection arriving at it. Note `?person` is in the **object** position inside the `NOT EXISTS` — you're testing for a missing *incoming* arrow, the mirror of B4's missing *outgoing* one.

</details>

### C5 — Stretch: grandchildren count per grandparent  *(a chain feeding a count)*

Everything at once: the two-hop connection from C1, gathered and counted like B3.

**Trace the connections.** For each person, follow `parentOf/parentOf` and count where you land. Who are the grandparents, and how many grandchildren each? (Ada and Ben are the couple above Cara and Dev.)

**Now write it.** `ex:parentOf/ex:parentOf` to reach grandchildren, `GROUP BY` the grandparent, `COUNT(DISTINCT ?gc)`. Ask yourself why `DISTINCT` earns its place here.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?gpname (COUNT(DISTINCT ?gc) AS ?n) WHERE {
    # ?gp --ex:parentOf/ex:parentOf--> ?gc ;  ?gp --foaf:name--> ?gpname
}
# GROUP BY ... ORDER BY ...

<details>
<summary>Solution</summary>

**Path traced:** Ada `parentOf/parentOf` → Eli, Fay, Gus (3). Ben → the same three (3). Nobody else is two hops above anyone. **Ada 3, Ben 3.**

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?gpname (COUNT(DISTINCT ?gc) AS ?n) WHERE {
    ?gp ex:parentOf/ex:parentOf ?gc .
    ?gp foaf:name ?gpname .
}
GROUP BY ?gpname
ORDER BY DESC(?n) ?gpname
```

| ?gpname | ?n |
|---|---|
| Ada | 3 |
| Ben | 3 |

`COUNT(DISTINCT ?gc)` counts distinct grandchildren, not distinct *paths* to them — the same route-vs-node distinction as C3, now inside an aggregate. One line of connection-tracing (`/`) feeding one line of counting (`GROUP BY`) is the whole lab in miniature.

</details>

---

## What you actually practised

Not "SPARQL syntax" — **reading a graph as connections and copying the path into a query.** Every pattern maps back to something you traced by hand:

| When you traced… | You wrote… | First seen |
|---|---|---|
| one arrow to a value | one triple pattern | A1 |
| several arrows out of one node | several patterns, same subject `?b` | A2 |
| a hand-off through a shared node | a repeated variable (the join) | A3 |
| in / out of a middle link node | the enrolment pivot `?e` | B1–B2 |
| an arrow that might be missing | `OPTIONAL { }` | B2 |
| gathering connections and tallying | `GROUP BY` + `COUNT` | B3 |
| a node with no arrow in/out | `FILTER NOT EXISTS` | B4, C4 |
| the same arrow twice / many times | path `/` and `+` (`*`) | C1, C2 |
| two arrows from one shared node | self-join + `DISTINCT` | C3, C5 |

**The one habit to keep:** before you type a query, draw the path — starting node, each arrow, landing node. Then translate arrow-by-arrow. When your hand answer and the engine disagree, re-trace the arrows until you find the connection you misread. That gap is the whole point.

Ready for a bigger graph — multiple files, inference, a live Wikidata endpoint? Continue to **Lab 9 — SPARQL: Querying the Movie Knowledge Graph**.